In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt 
import os
import sys
sys.path.append(os.path.abspath(os.path.join('..')))
from megatron.checkpointing import model_diff, sparsification, csr_sparsification

Zarr-based strategies will not be registered because of missing packages
/home/pengyanxin/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ckpt_path2 = "/mnt/pengyanxin/my_megatron/examples/gpt3/gpt2_345m/iter_0000909/mp_rank_00/model_optim_rng.pt"
ckpt_path1 = "/mnt/pengyanxin/my_megatron/examples/gpt3/gpt2_345m/iter_0000910/mp_rank_00/model_optim_rng.pt"
ckpt1 = torch.load(ckpt_path1, map_location='cpu')
ckpt2 = torch.load(ckpt_path2, map_location='cpu')


model_state_dict1 = ckpt1['model']
model_state_dict2 = ckpt2['model']


In [3]:
print(model_state_dict1['language_model']['embedding'])


{'word_embeddings': OrderedDict([('weight', tensor([[ 2.3483e-02,  3.8940e-02,  7.0374e-02,  ..., -9.3262e-02,
         -1.4844e-01, -1.0388e-01],
        [ 1.6312e-02,  2.3514e-02,  4.4632e-03,  ..., -2.8030e-02,
         -9.6436e-02, -1.2253e-02],
        [ 2.3071e-02,  4.3060e-02,  1.1505e-02,  ..., -4.7668e-02,
         -1.0870e-01, -5.2368e-02],
        ...,
        [-2.0889e-02,  1.3138e-02, -8.0338e-03,  ...,  2.4597e-02,
          2.1301e-02, -1.4186e-05],
        [-2.5620e-02, -2.5272e-03, -4.5395e-03,  ...,  1.2474e-02,
          1.7548e-02, -2.4719e-02],
        [-4.2236e-02,  1.8723e-02, -2.7557e-02,  ...,  4.4403e-02,
          1.4412e-02, -2.9831e-02]], dtype=torch.float16))]), 'position_embeddings': OrderedDict([('weight', tensor([[ 0.0112,  0.0222, -0.0295,  ...,  0.0124, -0.0004,  0.0129],
        [ 0.0339, -0.0025,  0.0216,  ..., -0.0034, -0.0095,  0.0253],
        [ 0.0174,  0.0040,  0.0076,  ...,  0.0063, -0.0178,  0.0331],
        ...,
        [-0.0165,  0.0009,  0

In [4]:

diff = {}
model_diff(model_state_dict1, model_state_dict2, diff)

print(diff)

{'language_model': {'embedding': {'word_embeddings': {'weight': tensor([[ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
          0.0000e+00,  0.0000e+00],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
          0.0000e+00, -7.6294e-06],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
          0.0000e+00,  0.0000e+00],
        ...,
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
          0.0000e+00,  0.0000e+00],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
          0.0000e+00,  0.0000e+00],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
          0.0000e+00,  0.0000e+00]], dtype=torch.float16)}, 'position_embeddings': {'weight': tensor([[ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
         -1.1921e-06,  0.0000e+00],
        [ 0.0000e+00, -1.9073e-06,  0.0000e+00,  ..., -1.9073e-06,
          0.0000e+00,  0.0000e+00],
        [ 0.0000e+00,  0.0000e+00,  0.000

In [5]:
def generate_bitmask(delta, bitmasks):
    for key, value in delta.items():
        if isinstance(value, dict):
            bitmasks[key] = {}
            generate_bitmask(value, bitmasks[key])
        elif isinstance(value, torch.Tensor):
            bitmasks[key] = (value != 0).to(torch.uint8)  # Use uint8 instead of bool
        elif value is None:
            bitmasks[key] = None
        else:
            raise ValueError(f"Unsupported type for key '{key}': {type(value)}")

In [6]:
def flatten_tensor(tensor):
    return tensor.view(-1)

def store_sparse_delta_with_bitmask(delta, bitmask, file_path):
    def process_delta_and_bitmask(delta, bitmask):
        sparse_delta_with_bitmask = {}
        for key in delta.keys():
            if isinstance(delta[key], dict):
                sparse_delta_with_bitmask[key] = process_delta_and_bitmask(delta[key], bitmask[key])
            elif isinstance(delta[key], torch.Tensor):
                flattened_delta = flatten_tensor(delta[key])
                flattened_bitmask = flatten_tensor(bitmask[key])
                sparse_values = flattened_delta[flattened_bitmask == 1]
                sparse_delta_with_bitmask[key] = (sparse_values, flattened_bitmask)
            elif delta[key] is None:
                sparse_delta_with_bitmask[key] = None
            else:
                raise ValueError(f"Unsupported type for key '{key}': {type(delta[key])}")
        return sparse_delta_with_bitmask

    sparse_delta_with_bitmask = process_delta_and_bitmask(delta, bitmask)
    torch.save(sparse_delta_with_bitmask, file_path)

def load_sparse_delta_with_bitmask(file_path):
    return torch.load(file_path)

# Store the sparse delta and bitmask
bitmasks = {}
generate_bitmask(diff, bitmasks)
store_sparse_delta_with_bitmask(diff, bitmasks, 'sparse_delta.pt')

In [7]:
def load_sparse_delta_with_bitmask(file_path):
    return torch.load(file_path)

def reconstruct_checkpoint_with_bitmask(base_checkpoint, sparse_delta_with_bitmask):
    def process_reconstruction(base_checkpoint, sparse_delta_with_bitmask):
        new_checkpoint = {}
        for key in base_checkpoint.keys():
            if key in sparse_delta_with_bitmask:
                if isinstance(sparse_delta_with_bitmask[key], tuple):
                    sparse_values, flattened_bitmask = sparse_delta_with_bitmask[key]
                    delta = torch.zeros_like(base_checkpoint[key]).view(-1)
                    delta[flattened_bitmask == 1] = sparse_values
                    delta = delta.view(base_checkpoint[key].shape)
                    new_checkpoint[key] = base_checkpoint[key] + delta
                elif isinstance(sparse_delta_with_bitmask[key], dict):
                    new_checkpoint[key] = process_reconstruction(base_checkpoint[key], sparse_delta_with_bitmask[key])
                elif sparse_delta_with_bitmask[key] is None:
                    new_checkpoint[key] = base_checkpoint[key]
                else:
                    raise ValueError(f"Unsupported type for key '{key}' in sparse_delta_with_bitmask: {type(sparse_delta_with_bitmask[key])}")
            else:
                new_checkpoint[key] = base_checkpoint[key]
        return new_checkpoint

    return process_reconstruction(base_checkpoint, sparse_delta_with_bitmask)

recovered_state_dict = reconstruct_checkpoint_with_bitmask(model_state_dict1, load_sparse_delta_with_bitmask('sparse_delta.pt'))
print(recovered_state_dict)





{'language_model': {'embedding': {'word_embeddings': {'weight': tensor([[ 2.3483e-02,  3.8940e-02,  7.0374e-02,  ..., -9.3262e-02,
         -1.4844e-01, -1.0388e-01],
        [ 1.6312e-02,  2.3514e-02,  4.4632e-03,  ..., -2.8030e-02,
         -9.6436e-02, -1.2260e-02],
        [ 2.3071e-02,  4.3060e-02,  1.1505e-02,  ..., -4.7668e-02,
         -1.0870e-01, -5.2368e-02],
        ...,
        [-2.0889e-02,  1.3138e-02, -8.0338e-03,  ...,  2.4597e-02,
          2.1301e-02, -1.4186e-05],
        [-2.5620e-02, -2.5272e-03, -4.5395e-03,  ...,  1.2474e-02,
          1.7548e-02, -2.4719e-02],
        [-4.2236e-02,  1.8723e-02, -2.7557e-02,  ...,  4.4403e-02,
          1.4412e-02, -2.9831e-02]], dtype=torch.float16)}, 'position_embeddings': {'weight': tensor([[ 0.0112,  0.0222, -0.0295,  ...,  0.0124, -0.0004,  0.0129],
        [ 0.0339, -0.0025,  0.0216,  ..., -0.0034, -0.0095,  0.0253],
        [ 0.0174,  0.0040,  0.0076,  ...,  0.0063, -0.0178,  0.0331],
        ...,
        [-0.0165,  0.000

In [8]:
test = {}
model_diff(model_state_dict2, recovered_state_dict, test)
print(test)

{'language_model': {'embedding': {'word_embeddings': {'weight': tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]], dtype=torch.float16)}, 'position_embeddings': {'weight': tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]], dtype=torch.float16)}}, 'encoder': {'layers.0.input_norm.weight': tensor([0., 0., 0.,  ..., 0., 0., 0.], dtype=torch.float16), 'layers.0.input_norm.bias': tensor([0., 0., 0.,  ..., 0., 0., 0.], dtype=torch.float16), 'layers.0.self_attention.query_key_value.weight': tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ...

In [17]:
print(bitmasks['language_model']['encoder']['layers.0.mlp.dense_4h_to_h.weight'].shape)

torch.Size([1024, 4096])


In [10]:
sparse_model_dict = {}
sparsification(diff, sparse_model_dict)



In [11]:
print(sparse_model_dict)


{'language_model': {'embedding': {'word_embeddings': {'weight': {'size': torch.Size([50304, 1024]), 'compressed_indices': array([[    0,     0,     0, ..., 50300, 50300, 50303],
       [   14,    15,    17, ...,    91,   633,   215]], dtype=uint16), 'values': tensor([-1.9073e-06,  1.9073e-06, -1.5259e-05,  ..., -4.7684e-07,
         1.1921e-07,  7.6294e-06], dtype=torch.float16)}}, 'position_embeddings': {'weight': {'size': torch.Size([1024, 1024]), 'compressed_indices': array([[   0,    0,    0, ..., 1023, 1023, 1023],
       [   4,    7,   17, ..., 1012, 1018, 1020]], dtype=uint16), 'values': tensor([-3.8147e-06,  3.8147e-06, -2.8014e-06,  ...,  2.8610e-06,
        -3.8147e-06, -9.5367e-07], dtype=torch.float16)}}}, 'encoder': {'layers.0.input_norm.weight': {'size': torch.Size([1024]), 'compressed_indices': array([[  4,  81, 147, 169, 253, 538, 977]], dtype=uint16), 'values': tensor([ 0.0002,  0.0002,  0.0002,  0.0002,  0.0002, -0.0002,  0.0002],
       dtype=torch.float16)}, 'layers

In [12]:
torch.save(sparse_model_dict, "sparse_model.pt")

In [13]:
def tensors_to_numpy(data):
    if isinstance(data, dict):
        return {key: tensors_to_numpy(value) for key, value in data.items()}
    elif isinstance(data, torch.Tensor):
        return data.cpu().numpy()
    else:
        return data

def calculate_percentage_of_changes(tensor_diff):
    # Flatten the tensor to a 1D array
    if tensor_diff is None:
        return 0.0
    
    tensor_diff_flat = tensor_diff.flatten()
    
    # Count the number of non-zero elements
    num_non_zero = (tensor_diff_flat != 0).sum()
    
    # Calculate the total number of elements
    total_elements = tensor_diff_flat.size
    
    # Compute the percentage of changed elements
    percentage_changed = (num_non_zero / total_elements) * 100
    
    return percentage_changed

diff_state_dict_numpy = tensors_to_numpy(diff)

total_elements = 0 
total_percentage = 0
for key in diff_state_dict_numpy['language_model']['encoder'].keys():
    tensor_diff_to_plot = diff_state_dict_numpy['language_model']['encoder'][key]
    percentage_changed = calculate_percentage_of_changes(tensor_diff_to_plot)
    print(f"Percentage of changed elements for key '{key}': {percentage_changed:.2f}%")
    total_percentage += percentage_changed
    total_elements += 1

print(f"Average percentage of changed elements: {total_percentage / total_elements:.2f}%")

Percentage of changed elements for key 'layers.0.input_norm.weight': 0.68%
Percentage of changed elements for key 'layers.0.input_norm.bias': 2.73%
Percentage of changed elements for key 'layers.0.self_attention.query_key_value.weight': 14.97%
Percentage of changed elements for key 'layers.0.self_attention.query_key_value.bias': 27.28%
Percentage of changed elements for key 'layers.0.self_attention.query_key_value._extra_state': 0.00%
Percentage of changed elements for key 'layers.0.self_attention.dense.weight': 30.65%
Percentage of changed elements for key 'layers.0.self_attention.dense.bias': 51.37%
Percentage of changed elements for key 'layers.0.self_attention.dense._extra_state': 0.00%
Percentage of changed elements for key 'layers.0.post_attention_norm.weight': 0.29%
Percentage of changed elements for key 'layers.0.post_attention_norm.bias': 7.03%
Percentage of changed elements for key 'layers.0.mlp.dense_h_to_4h.weight': 12.97%
Percentage of changed elements for key 'layers.0.ml

In [14]:
diff_state_dict = {}
def iterate_and_subtract(dict1, dict2, diff_dict):
    for key in dict1.keys(): 
        if key in dict2.keys():
            if isinstance(dict1[key], dict) and isinstance(dict2[key], dict):
                diff_dict[key] = {}
                iterate_and_subtract(dict1[key], dict2[key], diff_dict[key])
            elif isinstance(dict1[key], torch.Tensor) and isinstance(dict2[key], torch.Tensor):
                diff_dict[key] = dict1[key] - dict2[key]
                max_diff_index = torch.argmax(torch.abs(diff_dict[key]))
                max_diff_indices = torch.unravel_index(max_diff_index, diff_dict[key].shape)
                dict1_value = dict1[key][max_diff_indices]
                dict2_value = dict2[key][max_diff_indices]
                max_diff_value = diff_dict[key][max_diff_indices]
                print(f"Max diff for key '{max_diff_indices}': {max_diff_value}")
                print(f"Value in dict1 for key '{max_diff_indices}: {dict1_value}")
                print(f"Value in dict1 for key '{max_diff_indices}: {dict2_value}")
            else:
                raise ValueError(f"Mismatched types for key '{key}': {type(dict1[key])} vs {type(dict2[key])}")
        else:
            raise ValueError(f"Key '{key}' not found in second dict")

iterate_and_subtract(state_dict1, state_dict2, diff_state_dict)

print(diff_state_dict['language_model']['encoder'].keys())

print(diff_state_dict)

NameError: name 'state_dict1' is not defined

In [ ]:
def tensors_to_numpy(data):
    if isinstance(data, dict): 
        return {key: tensors_to_numpy(value) for key, value in data.items()}
    elif isinstance(data, torch.Tensor): 
        if data.dtype == torch.bfloat16:
            data = data.to(torch.float16)
        return data.cpu().numpy()
    else: 
        return data
    
diff_state_dict_numpy = tensors_to_numpy(diff_state_dict)

In [ ]:
def plot_scatter_for_row_or_column(tensor_diff, index=0, axis=0, title="Scatter Plot of Differences", max_dots=1000):
    if axis == 0:
        data = tensor_diff[index, :]
    else:
        data = tensor_diff[:, index]
    
    data = np.array(data)
    
    plt.figure(figsize=(10, 6))
    plt.scatter(range(len(data[:max_dots])), data[:max_dots], alpha=0.6)
    plt.title(title)
    plt.xlabel("Index")
    plt.ylabel("Value")
    plt.show()
diff_tensor = diff_state_dict_numpy['language_model']['encoder']['layers.0.mlp.dense_h_to_4h.weight'] 
plot_scatter_for_row_or_column(diff_tensor, index=3525, axis=0, title="Scatter Plot of Row Differences", max_dots=1000)

In [ ]:
def plot_non_zero_heatmap(tensor_diff, title="Non-zero Differences Heatmap"):
    # Create a mask for non-zero values
    non_zero_mask = tensor_diff != 0
    
    # Plot the heatmap
    plt.figure(figsize=(10, 8))
    sns.heatmap(non_zero_mask, cbar=False, cmap='viridis')
    plt.title(title)
    plt.xlabel("D1: rows of the weight matrix")
    plt.ylabel("D2: columns of the weight matrix")
    plt.show()



In [ ]:
# Example of accessing and plotting a specific tensor difference
key_to_plot = 'language_model.encoder.layers.0.self_attention.query_key_value.weight'
tensor_diff_to_plot = diff_state_dict_numpy['language_model']['encoder']['layers.0.mlp.dense_h_to_4h.weight']

# Plotting the heatmap of non-zero differences
plot_non_zero_heatmap(tensor_diff_to_plot, title=key_to_plot)


In [ ]:
print(tensor_diff_to_plot.shape)

In [ ]:
def calculate_percentage_of_changes(tensor_diff):
    # Flatten the tensor to a 1D array
    tensor_diff_flat = tensor_diff.flatten()
    
    # Count the number of non-zero elements
    num_non_zero = (tensor_diff_flat != 0).sum()
    
    # Calculate the total number of elements
    total_elements = tensor_diff_flat.size
    
    # Compute the percentage of changed elements
    percentage_changed = (num_non_zero / total_elements) * 100
    
    return percentage_changed

# Example usage
for key in diff_state_dict_numpy['language_model']['encoder'].keys():
    tensor_diff_to_plot = diff_state_dict_numpy['language_model']['encoder'][key]
    percentage_changed = calculate_percentage_of_changes(tensor_diff_to_plot)
    print(f"Percentage of changed elements for key '{key}': {percentage_changed:.2f}%")



In [ ]:
def tensors_to_numpy(data):
    if isinstance(data, dict): 
        return {key: tensors_to_numpy(value) for key, value in data.items()}
    elif isinstance(data, torch.Tensor): 
        if data.dtype == torch.bfloat16:
            data = data.to(torch.float16)
        return data.cpu().numpy()
    else: 
        return data
    
diff_state_dict_numpy = tensors_to_numpy(diff_state_dict)

In [ ]:
def find_max_change(diff_dict):
    max_change = 0
    max_change_key = None

    def recursive_find_max_change(d):
        nonlocal max_change, max_change_key
        for key, value in d.items():
            if isinstance(value, dict):
                recursive_find_max_change(value)
            elif isinstance(value, np.ndarray):
                current_max_change = np.max(np.abs(value))
                if current_max_change > max_change:
                    max_change = current_max_change
                    max_change_key = key
            else:
                raise ValueError(f"Unsupported type for key '{key}': {type(value)}")

    recursive_find_max_change(diff_dict)
    return max_change, max_change_key

# Example usage
max_change, max_change_key = find_max_change(diff_state_dict_numpy)
print(f"Maximum change: {max_change} in key: {max_change_key}")
print(diff_state_dict_numpy)